In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/10011
1


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  331.1350611509488
RUN  2 , total integrated cost =  222.2127650671246
RUN  3 , total integrated cost =  119.99259760188492
RUN  4 , total integrated cost =  114.40435285954557
RUN  5 , total integrated cost =  106.61832230800711
RUN  6 , total integrated cost =  103.77499316644568
RUN  7 , total integrated cost =  100.46484981630651
RUN  8 , total integrated cost =  98.90799158598757
RUN  9 , total integrated cost =  96.98146475876757
RUN  10 , total integrated cost =  95.98948343823976
RUN  11 , total integrated cost =  94.8863737630328
RUN  12 , total integrated cost =  93.40699600280452
RUN  13 , total integrated cost =  91.84662610204016
RUN  14 , total integrated cost =  90.98216009199413
RUN  15 , total integrated cost =  89.9564904764613
RUN  16 ,

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  603 , total integrated cost =  74.725356625733
Improved over  603  iterations in  68.07995091937482  seconds by  98.53401789687669  percent.
Problem in initial value trasfer:  Vmean_exc -67.7990089727647 -67.80291268925316
weight =  682.1365676084818
set cost params:  1.0 0.0 682.1365676084818
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5093.784535112272
Gradient descend method:  None
RUN  1 , total integrated cost =  5082.096024585435
RUN  2 , total integrated cost =  5082.096024585433
RUN  3 , total integrated cost =  5082.09602458543


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  5082.096024585425
RUN  5 , total integrated cost =  5082.096024585425
Control only changes marginally.
RUN  5 , total integrated cost =  5082.096024585425
Improved over  5  iterations in  0.6005730926990509  seconds by  0.2294661355673071  percent.
Problem in initial value trasfer:  Vmean_exc -66.99623699101885 -67.01534116212277
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  9290.719474796228
RUN  2 , total integrated cost =  9239.861514102067
RUN  3 , total integrated cost =  9238.843953397334
RUN  4 , total integrated cost =  9238.781702577025
RUN  5 , total integrated cost =  9238.75271708068
RUN  6 , total integrated cost =  9238.732049046703
RUN  7 , total integrated cost =  9238.714662455837
RUN  8 , total integrated cost =  9238.698673545463
RUN  9 , total i

...

RUN  10000 , total integrated cost =  22499.03328660732
RUN  10000 , total integrated cost =  22499.03328660732
Improved over  10000  iterations in  713.0285746790469  seconds by  0.03746274674375627  percent.
Problem in initial value trasfer:  Vmean_exc -56.69910613882232 -56.699167614338286
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  32423.295786314953
RUN  2 , total integrated cost =  32304.82064465916
RUN  3 , total integrated cost =  32301.43639076027
RUN  4 , total integrated cost =  32301.351747292345
RUN  5 , total integrated cost =  32301.347419774047
RUN  6 , total integrated cost =  32301.347148099827
RUN  7 , total integrated cost =  32301.34714542252
RUN  8 , total integrated cost =  32301.347018641274
RUN  9 , total integrated cost =  32301.340541336867
RUN  10 , total integrated co

ERROR:root:Problem in initial value trasfer


RUN  18 , total integrated cost =  32301.334182666855
Improved over  18  iterations in  1.2874567117542028  seconds by  2.970008277681984  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383082211083 -56.70383671731973
weight =  10.306091778940022
set cost params:  1.0 0.0 10.306091778940022
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32304.76306799628
Gradient descend method:  None
RUN  1 , total integrated cost =  32304.763067996275


ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  32304.763067996275
Control only changes marginally.
RUN  2 , total integrated cost =  32304.763067996275
Improved over  2  iterations in  0.32319948449730873  seconds by  1.4210854715202004e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383082211083 -56.70383671731973


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
[0, 5] []
closest index  5
set 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  399 , total integrated cost =  4372.34942540425
Improved over  399  iterations in  25.3752181250602  seconds by  52.31864601481623  percent.
Problem in initial value trasfer:  Vmean_exc -56.62773685293829 -56.627731572617456
weight =  20.838811365970578
set cost params:  1.0 0.0 20.838811365970578
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4760.304249713554
Gradient descend method:  None
RUN  1 , total integrated cost =  4760.304249713554
Control only changes marginally.
RUN  1 , total integrated cost =  4760.304249713554
Improved over  1  iterations in  0.17789296060800552  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62773685293829 -56.627731572617456
-------  15 0.4500000000000001 0.4500000000000002
[0, 5] []
closest index  5
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13081.865440048972
Gradient descend met

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  103 , total integrated cost =  9244.457173780278
Improved over  103  iterations in  6.752517513930798  seconds by  29.333800166762217  percent.
Problem in initial value trasfer:  Vmean_exc -56.64465155604278 -56.64508584700779
weight =  14.082032504049188
set cost params:  1.0 0.0 14.082032504049188
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9348.300235036992
Gradient descend method:  None
RUN  1 , total integrated cost =  9348.300235036992
Control only changes marginally.
RUN  1 , total integrated cost =  9348.300235036992
Improved over  1  iterations in  0.1788272112607956  seconds by  0.0  percent.


...

Control only changes marginally.
RUN  119 , total integrated cost =  9088.585016927618
Improved over  119  iterations in  7.643677990883589  seconds by  10.215601757085139  percent.
Problem in initial value trasfer:  Vmean_exc -56.6442791858382 -56.64444750293321
weight =  11.02478383589958
set cost params:  1.0 0.0 11.02478383589958
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9101.447129087652
Gradient descend method:  None
RUN  1 , total integrated cost =  9101.447129087652
Control only changes marginally.
RUN  1 , total integrated cost =  9101.447129087652
Improved over  1  iterations in  0.1787766721099615  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6442791858382 -56.64444750293321
-------  145 0.5750000000000002 0.9000000000000006
[0, 5] [5]
closest index  0
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33401.20248625345
Gradient descend method

ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  32282.19990228664
RUN  11 , total integrated cost =  32282.19990228664
Control only changes marginally.
RUN  11 , total integrated cost =  32282.19990228664
Improved over  11  iterations in  0.8524886183440685  seconds by  3.3501865222587384  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383239538773 -56.703835475427596
weight =  10.312200397631416
set cost params:  1.0 0.0 10.312200397631416
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32285.103450361228
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32285.103450361228
Control only changes marginally.
RUN  1 , total integrated cost =  32285.103450361228
Improved over  1  iterations in  0.1819971166551113  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70383239538773 -56.703835475427596
------------------------------------------------------------
-------------------- 2
------------------------------------------------------------
found solution:  [0, 5]
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  15 0.4500000000000001 0.4500000000000002
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  20 0.4500000000000001 0.4750000000000002
closest index  -1
set cost params:  1.0 0.0 10.0
all options tried already
-------  25 0.4250000000000001 0.5000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  683.1759326649932
set cost params:  1.0 0.0 683.1759326649932
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5089.833302122142
Gradient descend method:  None
RUN  1 , total integrated cost =  5089.833302122142
Control only changes marginally.


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5089.833302122142
Improved over  1  iterations in  0.1794960591942072  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -66.99623699101885 -67.01534116212277
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  30 0.4250000000000001 0.5250000000000002
converged for  30
-------  35 0.5500000000000003 0.5250000000000002
converged for  35
-------  40 0.5250000000000001 0.5500000000000003
converged for  40
-------  45 0.5000000000000002 0.5750000000000003
no convergence
weight =  12.15826501233706
set cost params:  1.0 0.0 12.15826501233706
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  18009.483933778738
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  18009.483933778738
Control only changes marginally.
RUN  1 , total integrated cost =  18009.483933778738
Improved over  1  iterations in  0.1810097973793745  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.689377172277936 -56.689621023415235
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
no convergence
weight =  10.86148376876805
set cost params:  1.0 0.0 10.86148376876805
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  27364.742172945425
Gradient descend method:  None


...

RUN  1 , total integrated cost =  13518.97958008113
Control only changes marginally.
RUN  1 , total integrated cost =  13518.97958008113
Improved over  1  iterations in  0.18190392293035984  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.671637411274496 -56.67185902419228
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
no convergence
weight =  9.99865681218412
set cost params:  1.0 0.0 9.99865681218412
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  32312.790022718145
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  32312.790022718145
Control only changes marginally.
RUN  1 , total integrated cost =  32312.790022718145
Improved over  1  iterations in  0.1797778569161892  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70384035390257 -56.703841668840255
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  130 0.6000000000000003 0.8500000000000005
no convergence
weight =  9.681807815925723
set cost params:  1.0 0.0 9.681807815925723
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  37466.06414979616
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  37466.06414979616
Control only changes marginally.
RUN  1 , total integrated cost =  37466.06414979616
Improved over  1  iterations in  0.18126614391803741  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70116848156453 -56.701147315049845
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
[[True, False], [False, False], [True, False], [True, False], [True, False], [True, False], [True, False], [True, False], [True, False], [False, False], [True, False], [True, False], [False, False], [True, False], [True, False], [False, False], [True, False], [False, False], [True, False], [True, False], [True, False], [False, False], [True, False], [True, False], [True, False], [True, False], [False, False], [True, False], [True, False], [True, False]]
-------  0 0.4000000000000001 0

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  78.55358915041748
Gradient descend method:  None
RUN  1 , total integrated cost =  74.46797665393414
RUN  2 , total integrated cost =  74.36879087161901
RUN  3 , total integrated cost =  74.32651335931803
RUN  4 , total integrated cost =  74.23165937946469
RUN  5 , total integrated cost =  74.19108593902575
RUN  6 , total integrated cost =  74.05640128658254
RUN  7 , total integrated cost =  74.0144166640216
RUN  8 , total integrated cost =  73.8395949960971
RUN  9 , total integrated cost =  68.88535884385259
RUN  10 , total integrated cost =  6.38533410709001
RUN  11 , total integrated cost =  5.859484911897436
RUN  12 , total integrated cost =  5.854918860229946
RUN  13 , total integrated cost =  5.854663874907446
RUN  14 , total integrated cost =  5.854557893170217
RUN  15 , total integrated cost =  5.854483321198438
RUN  16 , total

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  45 , total integrated cost =  5.815483979812745
Improved over  45  iterations in  9.439670529216528  seconds by  92.59679405777752  percent.
Problem in initial value trasfer:  Vmean_exc -67.93395809113018 -67.93676019361277
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  58.13413480882925
Gradient descend method:  HS
RUN  1 , total integrated cost =  58.10873172251402
RUN  2 , total integrated cost =  58.10121247183431
RUN  3 , total integrated cost =  58.10051778857979
RUN  4 , total integrated cost =  58.10051778857969
RUN  5 , total integrated cost =  58.100517788579644


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  58.100517788579644
Control only changes marginally.
RUN  6 , total integrated cost =  58.100517788579644
Improved over  6  iterations in  2.317930394783616  seconds by  0.057826645842681046  percent.
Problem in initial value trasfer:  Vmean_exc -67.83950820769466 -67.84292891203927
weight =  8772.226164262613
set cost params:  1.0 0.0 8772.226164262613
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5095.144806018708
Gradient descend method:  None
RUN  1 , total integrated cost =  5089.904848440343
RUN  2 , total integrated cost =  5089.904844095281
RUN  3 , total integrated cost =  5089.904844095276
RUN  4 , total integrated cost =  5089.9048440952665
RUN  5 , total integrated cost =  5089.904844095264


...

Control only changes marginally.
RUN  296 , total integrated cost =  52.63014779283971
Improved over  296  iterations in  109.79926787689328  seconds by  99.83696852150057  percent.
Problem in initial value trasfer:  Vmean_exc -65.03875274150941 -65.05432090649491
weight =  100
set cost params:  1.0 0.0 100.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  521.8657517351357
Gradient descend method:  HS
RUN  1 , total integrated cost =  516.3580300359529
RUN  2 , total integrated cost =  516.2049896260343
RUN  3 , total integrated cost =  516.2049896260338


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  516.2049896260338
Control only changes marginally.
RUN  4 , total integrated cost =  516.2049896260338
Improved over  4  iterations in  2.9595829527825117  seconds by  1.0847161535856031  percent.
Problem in initial value trasfer:  Vmean_exc -62.801884624609286 -62.81753942554383
weight =  6447.998389379148
set cost params:  1.0 0.0 6447.998389379148
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33041.764862162934
Gradient descend method:  None
RUN  1 , total integrated cost =  32608.436503827732
RUN  2 , total integrated cost =  32607.09745350727
RUN  3 , total integrated cost =  32606.65261546685
RUN  4 , total integrated cost =  32606.188919755743
RUN  5 , total integrated cost =  32606.075207987593
RUN  6 , total integrated cost =  32605.923356565938
RUN  7 , total integrated cost =  32605.85556169668
RUN  8 , total integrated cost =  32605.73087284201
RUN  9 , total integrated cost =  32605.656174479576
RUN  10 , total in

ERROR:root:Problem in initial value trasfer


RUN  60 , total integrated cost =  32596.698828794848
Control only changes marginally.
RUN  60 , total integrated cost =  32596.698828794848
Improved over  60  iterations in  33.43120910413563  seconds by  1.346980208910523  percent.
Problem in initial value trasfer:  Vmean_exc -57.5636169109968 -57.543883101100405


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  8783.953858152401
set cost params:  1.0 0.0 8783.953858152401
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5096.707478787506
Gradient descend method:  None
RUN  1 , total integrated cost =  5096.707478462843
RUN  2 , total integrated cost =  5096.707478462836
RUN  3 , total integrated cost =  5096.707478462833


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  5096.707478462833
Control only changes marginally.
RUN  4 , total integrated cost =  5096.707478462833
Improved over  4  iterations in  3.3469810113310814  seconds by  6.370257210619457e-09  percent.
Problem in initial value trasfer:  Vmean_exc -67.35169254259245 -67.36377324320176
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  5848.0052828942935
set cost params:  1.0 0.0 5848.0052828942935
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13015.614737990163
Gradient descend method:  None
RUN  1 , total integrated cost =  13015.6143970048
RUN  2 , total integrated cost =  13015.614394128743
RUN  3 , total integrated cost =  13015.614393899945
RUN  4 , total integrated cost =  13015.614393826389
RUN  5 , total integrated cost =  13015.614393799175
RUN  6 , total integrated cost =  13015.614393788988
RUN  7 , total integrated cost =  13015.614393785227
RUN  8 , total integrated cost =  13015.614393783791


ERROR:root:Problem in initial value trasfer


RUN  13 , total integrated cost =  13015.614393783448
Control only changes marginally.
RUN  13 , total integrated cost =  13015.614393783448
Improved over  13  iterations in  8.851831629872322  seconds by  2.6445674876640624e-06  percent.
Problem in initial value trasfer:  Vmean_exc -62.74519035074135 -62.77861316533141
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  6620.667585219695
set cost params:  1.0 0.0 6620.667585219695
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8230.641113533433
Gradient descend method:  None
RUN  1 , total integrated cost =  8230.641078866383


...

RUN  5 , total integrated cost =  34490.88801046107
Control only changes marginally.
RUN  5 , total integrated cost =  34490.88801046107
Improved over  5  iterations in  2.3056537806987762  seconds by  1.0099540759256342e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.703440327472954 -56.70341138448297
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 99
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [Tr

ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  34490.92243236996
Control only changes marginally.
RUN  5 , total integrated cost =  34490.92243236996
Improved over  5  iterations in  3.9071177635341883  seconds by  1.4394011316198885e-07  percent.
Problem in initial value trasfer:  Vmean_exc -56.70343931506075 -56.70341046694726
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 100
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [Tr

ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  34490.95559899138
Control only changes marginally.
RUN  5 , total integrated cost =  34490.95559899138
Improved over  5  iterations in  3.59802876226604  seconds by  8.762957293129148e-08  percent.
Problem in initial value trasfer:  Vmean_exc -56.703438567438766 -56.703409789393895
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.5250000000000001 0.8750000000000006
converged for  135
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
--------------- 101


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.1660698091571957
Gradient descend method:  None
RUN  1 , total integrated cost =  0.5934107231123723
RUN  2 , total integrated cost =  0.5934107231123692
RUN  3 , total integrated cost =  0.5934107231123688
RUN  4 , total integrated cost =  0.593410723112365


ERROR:root:Problem in initial value trasfer


RUN  5 , total integrated cost =  0.593410723112362
RUN  6 , total integrated cost =  0.593410723112362
Control only changes marginally.
RUN  6 , total integrated cost =  0.593410723112362
Improved over  6  iterations in  1.1461564898490906  seconds by  72.60426600270772  percent.
Problem in initial value trasfer:  Vmean_exc -67.93543785659834 -67.93823177855334
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  27.20026170478671
Gradient descend method:  None
RUN  1 , total integrated cost =  2.692122184491149
RUN  2 , total integrated cost =  2.6910680074122157
RUN  3 , total integrated cost =  2.6910260222425566
RUN  4 , total integrated cost =  2.6910260101119423
RUN  5 , total integrated cost =  2.6910260101119365
RUN  6 , total integrated cost =  2.691026010111933


ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  2.6910260101119325
RUN  8 , total integrated cost =  2.6910260101119325
Control only changes marginally.
RUN  8 , total integrated cost =  2.6910260101119325
Improved over  8  iterations in  1.2780294083058834  seconds by  90.10661721082498  percent.
Problem in initial value trasfer:  Vmean_exc -67.12816991393163 -67.1375921206126
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.040435326177429
Gradient descend method:  None
RUN  1 , total integrated cost =  1.3449266071964985
RUN  2 , total integrated cost =  1.3448827241006749
RUN  3 , total integrated cost =  1.3448827241006693
RUN  4 , total integrated cost =  1.3448827241006664


...

RUN  16 , total integrated cost =  3.605452984407937
RUN  17 , total integrated cost =  3.605452984407937
Control only changes marginally.
RUN  17 , total integrated cost =  3.605452984407937
Improved over  17  iterations in  2.5206819903105497  seconds by  88.01934545410464  percent.
Problem in initial value trasfer:  Vmean_exc -70.84168534867193 -70.87721738923764
-------  135 0.5250000000000001 0.8750000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  49.252835432607824
Gradient descend method:  None
RUN  1 , total integrated cost =  6.7108356048275
RUN  2 , total integrated cost =  6.698958624143574
RUN  3 , total integrated cost =  6.698642140719485
RUN  4 , total integrated cost =  6.698583483057718
RUN  5 , total integrated cost =  6.698555559685848
RUN  6 , total integrated cost =  6.698513514724282
RUN  7 , total integrated cost =  6.698017775562346
RUN  8 , total integrated cost =  6.698017775562277
RUN  9 , total

ERROR:root:Problem in initial value trasfer


RUN  10 , total integrated cost =  6.698017775562269
Control only changes marginally.
RUN  10 , total integrated cost =  6.698017775562269
Improved over  10  iterations in  1.5743435882031918  seconds by  86.40074684689554  percent.
Problem in initial value trasfer:  Vmean_exc -67.2913832536954 -67.32080108780775
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  110.80613946974172
Gradient descend method:  None
RUN  1 , total integrated cost =  13.012135679706216
RUN  2 , total integrated cost =  12.987588460117001
RUN  3 , total integrated cost =  12.980190827036918
RUN  4 , total integrated cost =  12.98016225249588
RUN  5 , total integrated cost =  12.980161862135155
RUN  6 , total integrated cost =  12.980161815400741
RUN  7 , total integrated cost =  12.980161815400697
RUN  8 , total integrated cost =  12.980161815400685
RUN  9 , total integrated cost =  12.980161815400642
RUN  1

ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  12.980161815400638
Control only changes marginally.
RUN  12 , total integrated cost =  12.980161815400638
Improved over  12  iterations in  1.694390894845128  seconds by  88.28570160686341  percent.
Problem in initial value trasfer:  Vmean_exc -63.701184183645765 -63.71748384817853


In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  0.593410723112362
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  0.593410723112362
Control only changes marginally.
RUN  1 , total integrated cost =  0.593410723112362
Improved over  1  iterations in  0.3009290974587202  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.93543785659834 -67.93823177855334
-------  15 0.4500000000000001 0.4500000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.6910260101119325
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  2.6910260101119325
Control only changes marginally.
RUN  1 , total integrated cost =  2.6910260101119325
Improved over  1  iterations in  0.3009556718170643  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.12816991393163 -67.1375921206126
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.3448827241006664
Gradient descend method:  None


...

RUN  1 , total integrated cost =  3.605452984407937
Control only changes marginally.
RUN  1 , total integrated cost =  3.605452984407937
Improved over  1  iterations in  0.30449281074106693  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -70.84168534867193 -70.87721738923764
-------  135 0.5250000000000001 0.8750000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6.698017775562269
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  6.698017775562269
Control only changes marginally.
RUN  1 , total integrated cost =  6.698017775562269
Improved over  1  iterations in  0.3055325448513031  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -67.2913832536954 -67.32080108780775
-------  145 0.5750000000000002 0.9000000000000006
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12.980161815400638
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  12.980161815400638
Control only changes marginally.
RUN  1 , total integrated cost =  12.980161815400638
Improved over  1  iterations in  0.30811840668320656  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -63.701184183645765 -63.71748384817853
--------------- 12
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
converged for  5
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
